
# 04 | Council Name and Classification Validation

## Project
Victorian Local Government Financial Sustainability
& Performance Intelligence (2020–2025)

## Purpose
Validate council identities and peer-group classifications
before cleaning the performance dataset.

## Objectives
1. Check council coverage across five financial years.
2. Identify inconsistent council names.
3. Investigate inconsistent group labels.
4. Compare councils against the category reference.
5. Validate standardised names and reporting coverage.
6. Prepare documented mapping rules for Silver.

## Input
- workspace.default.bronze_performance
- Council category data.csv

## Previous Work
- Notebook 01: Bronze ingestion completed.
- Notebook 02: Data quality assessment completed.
- Notebook 03: Schema and population validation completed.

## Expected Output
Validated council-name mapping and documented
classification issues for Notebook 05.

The original Bronze data remains unchanged.


In [0]:

from pyspark.sql import functions as F

df = spark.table("workspace.default.bronze_performance")

required_years = [
    "2020-21", "2021-22", "2022-23",
    "2023-24", "2024-25"
]

df = df.filter(F.col("FinancialYear").isin(required_years))

council_check = (
    df.groupBy("Council")
    .agg(
        F.countDistinct("FinancialYear").alias("Years_Present"),
        F.collect_set("FinancialYear").alias("Years")
    )
    .orderBy("Years_Present", "Council")
)

print("Bronze rows:", df.count())
print("Raw council names:", df.select("Council").distinct().count())

display(
    council_check.filter(F.col("Years_Present") < 5)
)


Bronze rows: 23068
Raw council names: 81


Council,Years_Present,Years
Moreland City,1,List(2020-21)
Colac-Otway Shire,2,"List(2021-22, 2020-21)"
Colac Otway Shire,3,"List(2024-25, 2023-24, 2022-23)"
Merri-bek City,4,"List(2024-25, 2023-24, 2022-23, 2021-22)"


In [0]:

group_check = (
    df.groupBy("Council")
    .agg(
        F.countDistinct("Group").alias("Group_Count"),
        F.collect_set("Group").alias("Group_Names")
    )
)

display(
    group_check
    .filter(F.col("Group_Count") > 1)
    .orderBy("Council")
)


Council,Group_Count,Group_Names
Alpine Shire,2,"List(Small Shire, SMALL_SHIRE)"
Ararat Rural City,2,"List(Small Shire, SMALL_SHIRE)"
Ballarat City,3,"List(Regional City, Regional, REGIONAL)"
Banyule City,2,"List(Metro, METRO)"
Bass Coast Shire,2,"List(Large Shire, LARGE_SHIRE)"
Baw Baw Shire,2,"List(Large Shire, LARGE_SHIRE)"
Bayside City,2,"List(Metro, METRO)"
Benalla Rural City,2,"List(Small Shire, SMALL_SHIRE)"
Boroondara City,2,"List(Metro, METRO)"
Borough of Queenscliffe,2,"List(Small Shire, SMALL_SHIRE)"


In [0]:

regional = df.filter(
    F.upper(F.trim(F.col("Group"))).isin(
        "REGIONAL", "REGIONAL CITY"
    )
)

display(
    regional
    .select("Council", "FinancialYear", "Group")
    .distinct()
    .orderBy("Council", "FinancialYear")
)


Council,FinancialYear,Group
Ballarat City,2020-21,REGIONAL
Ballarat City,2021-22,REGIONAL
Ballarat City,2022-23,Regional City
Ballarat City,2023-24,Regional
Ballarat City,2024-25,Regional City
Greater Bendigo City,2020-21,REGIONAL
Greater Bendigo City,2021-22,REGIONAL
Greater Bendigo City,2022-23,Regional City
Greater Bendigo City,2023-24,Regional
Greater Bendigo City,2024-25,Regional City


In [0]:

regional_check = regional.withColumn(
    "Group_Check",
    F.upper(F.trim(F.col("Group")))
)

both_groups = (
    regional_check
    .groupBy("Council")
    .agg(
        F.collect_set("Group_Check").alias("Categories"),
        F.countDistinct("Group_Check").alias("Category_Count")
    )
    .filter(F.col("Category_Count") > 1)
)

display(both_groups)


Council,Categories,Category_Count
Ballarat City,"List(REGIONAL CITY, REGIONAL)",2
Wodonga City,"List(REGIONAL CITY, REGIONAL)",2
Greater Bendigo City,"List(REGIONAL CITY, REGIONAL)",2
Mildura Rural City,"List(REGIONAL CITY, REGIONAL)",2
Warrnambool City,"List(REGIONAL CITY, REGIONAL)",2
Greater Geelong City,"List(REGIONAL CITY, REGIONAL)",2
Wangaratta Rural City,"List(REGIONAL CITY, REGIONAL)",2
Latrobe City,"List(REGIONAL CITY, REGIONAL)",2
Horsham Rural City,"List(REGIONAL CITY, REGIONAL)",2
Greater Shepparton,"List(REGIONAL CITY, REGIONAL)",2


In [0]:

categories = (
    spark.read
    .option("header", True)
    .csv(
        "/Volumes/workspace/default/council_raw/"
        "Council category data.csv"
    )
)

print("Reference records:", categories.count())

display(
    categories
    .groupBy("Council Category")
    .count()
    .orderBy("Council Category")
)

display(
    categories
    .filter(
        F.upper(F.col("Council Category")).contains("REGIONAL")
    )
    .orderBy("Council")
)


Reference records: 79


Council Category,count
Interface,9
Large shire,19
Metropolitan,22
Regional,10
Small shire,19


Council,Council Category
Ballarat City Council,Regional
Greater Bendigo City Council,Regional
Greater Geelong City Council,Regional
Greater Shepparton City Council,Regional
Horsham Rural City Council,Regional
Latrobe City Council,Regional
Mildura Rural City Council,Regional
Wangaratta Rural City Council,Regional
Warrnambool City Council,Regional
Wodonga City Council,Regional


In [0]:

from pyspark.sql import functions as F

validated = (
    df.withColumn("OriginalCouncil", F.col("Council"))
    .withColumn(
        "Council_Standard",
        F.when(
            F.col("Council") == "Moreland City",
            "Merri-bek City"
        )
        .when(
            F.col("Council") == "Colac-Otway Shire",
            "Colac Otway Shire"
        )
        .when(
            F.col("Council") == "Greater Shepparton",
            "Greater Shepparton City"
        )
        .otherwise(F.col("Council"))
    )
)

print("Original council names:",
      validated.select("OriginalCouncil").distinct().count())

print("Standardised councils:",
      validated.select("Council_Standard").distinct().count())

display(
    validated
    .filter(F.col("OriginalCouncil") != F.col("Council_Standard"))
    .select("OriginalCouncil", "Council_Standard")
    .distinct()
)


Original council names: 81
Standardised councils: 79


OriginalCouncil,Council_Standard
Greater Shepparton,Greater Shepparton City
Colac-Otway Shire,Colac Otway Shire
Moreland City,Merri-bek City


In [0]:

coverage = (
    validated.groupBy("Council_Standard")
    .agg(
        F.countDistinct("FinancialYear").alias("Years_Present")
    )
)

missing_years = coverage.filter(
    F.col("Years_Present") != 5
)

# Check whether standardisation creates duplicate keys
duplicate_keys = (
    validated
    .groupBy("Council_Standard", "FinancialYear", "ID")
    .count()
    .filter(F.col("count") > 1)
)

print("Standardised councils:", coverage.count())
print("Councils missing years:", missing_years.count())
print("Duplicate business keys:", duplicate_keys.count())

display(missing_years)
display(duplicate_keys)


Standardised councils: 79
Councils missing years: 0
Duplicate business keys: 0


Council_Standard,Years_Present


Council_Standard,FinancialYear,ID,count


In [0]:

from pyspark.sql import functions as F

def name_key(c):
    return F.regexp_replace(
        F.lower(F.trim(c)),
        r"\s+council$", ""
    )

def group_key(c):
    x = F.lower(F.trim(c))
    return (
        F.when(x.isin("metro", "metropolitan"), "Metropolitan")
        .when(x == "interface", "Interface")
        .when(x.isin("regional", "regional city"), "Regional")
        .when(x.isin("large shire", "large_shire"), "Large Shire")
        .when(x.isin("small shire", "small_shire"), "Small Shire")
        .otherwise(F.lit(None))
    )

source = (
    validated
    .withColumn("Council_Key", name_key(F.col("Council_Standard")))
    .withColumn("Group_Standard", group_key(F.col("Group")))
)

reference = (
    categories
    .withColumn("Council_Key", name_key(F.col("Council")))
    .withColumn(
        "Reference_Group",
        group_key(F.col("Council Category"))
    )
    .select("Council_Key", "Reference_Group")
)

source_names = source.select("Council_Key").distinct()
reference_names = reference.select("Council_Key").distinct()

print(
    "Unmatched source councils:",
    source_names.join(
        reference_names, "Council_Key", "left_anti"
    ).count()
)

print(
    "Unmatched reference councils:",
    reference_names.join(
        source_names, "Council_Key", "left_anti"
    ).count()
)

print(
    "Duplicate reference keys:",
    reference.groupBy("Council_Key")
    .count().filter("count > 1").count()
)

comparison = source.join(
    reference, "Council_Key", "left"
)

mismatches = comparison.filter(
    F.col("Reference_Group").isNull()
    | F.col("Group_Standard").isNull()
    | (F.col("Group_Standard") != F.col("Reference_Group"))
)

print("Rows with unmatched categories:", mismatches.count())

display(
    mismatches.select(
        "Council_Standard", "FinancialYear",
        "Group", "Group_Standard", "Reference_Group"
    ).distinct()
)


Unmatched source councils: 0
Unmatched reference councils: 0
Duplicate reference keys: 0
Rows with unmatched categories: 0


Council_Standard,FinancialYear,Group,Group_Standard,Reference_Group



## Results and Outcomes

### Dataset
- 23,068 performance records
- Financial years: 2020–21 to 2024–25
- 81 original council-name strings
- 79 standardised councils

### Validation Results
- Councils missing financial years: 0
- Duplicate council-year-indicator keys: 0
- Unmatched source councils: 0
- Unmatched reference councils: 0
- Duplicate reference keys: 0
- Category mismatches: 0

### Standardisation
- Moreland City → Merri-bek City
- Colac-Otway Shire → Colac Otway Shire
- Greater Shepparton → Greater Shepparton City
- 11 original group-label variants mapped to 5 categories

### Business Outcome
Validated consistent council identities and peer-group
classifications to support five-year performance trends,
financial sustainability analysis and Power BI benchmarking.

### Next Step
Notebook 05 — Council Data Cleaning (Silver Layer).
